# Bike Rental Data Processing Pipeline

In [4]:
import polars as pl
from pathlib import Path

# Project Paths

In [5]:
DATA_DIR = Path("../data")

DIRECT_RENTALS_PATH = DATA_DIR / "direct_pickup_bike_rentals.csv"
REGISTERED_RENTALS_PATH = DATA_DIR / "registered_bike_rentals.csv"
WEATHER_PATH = DATA_DIR / "weather.csv"
HOLIDAYS_PATH = DATA_DIR / "holidays.csv"

# Helper Functions

In [6]:
def load_csv(path: Path) -> pl.DataFrame:
    return pl.read_csv(path)

# Data Loading

In [7]:
direct_rentals = load_csv(DIRECT_RENTALS_PATH)
registered_rentals = load_csv(REGISTERED_RENTALS_PATH)
weather_df = load_csv(WEATHER_PATH)
holidays_df = load_csv(HOLIDAYS_PATH)

# Data Validation

In [8]:
def validate_dataframe(name: str, df: pl.DataFrame):

    print(f"\n{name}")
    print("-" * 50)

    print(f"Shape: {df.shape}")

    print("\nColumns:")
    print(df.columns)

    print("\nNull Counts:")
    print(df.null_count())

    print("\nDuplicate Rows:")
    print(df.is_duplicated().sum())

# Validate All Datasets

In [9]:
validate_dataframe(
    "Direct Rentals",
    direct_rentals
)

validate_dataframe(
    "Registered Rentals",
    registered_rentals
)

validate_dataframe(
    "Weather",
    weather_df
)

validate_dataframe(
    "Holidays",
    holidays_df
)


Direct Rentals
--------------------------------------------------
Shape: (620017, 4)

Columns:
['id', 'datetime', 'user_id', 'location_id']

Null Counts:
shape: (1, 4)
┌─────┬──────────┬─────────┬─────────────┐
│ id  ┆ datetime ┆ user_id ┆ location_id │
│ --- ┆ ---      ┆ ---     ┆ ---         │
│ u32 ┆ u32      ┆ u32     ┆ u32         │
╞═════╪══════════╪═════════╪═════════════╡
│ 0   ┆ 0        ┆ 0       ┆ 0           │
└─────┴──────────┴─────────┴─────────────┘

Duplicate Rows:
0

Registered Rentals
--------------------------------------------------
Shape: (2672662, 4)

Columns:
['id', 'datetime', 'user_id', 'location_id']

Null Counts:
shape: (1, 4)
┌─────┬──────────┬─────────┬─────────────┐
│ id  ┆ datetime ┆ user_id ┆ location_id │
│ --- ┆ ---      ┆ ---     ┆ ---         │
│ u32 ┆ u32      ┆ u32     ┆ u32         │
╞═════╪══════════╪═════════╪═════════════╡
│ 0   ┆ 0        ┆ 0       ┆ 0           │
└─────┴──────────┴─────────┴─────────────┘

Duplicate Rows:
0

Weather
--------

# Datetime Standardization

In [11]:
direct_rentals = (
    direct_rentals
    .with_columns(
        pl.col("datetime")
        .str.to_datetime()
    )
)

registered_rentals = (
    registered_rentals
    .with_columns(
        pl.col("datetime")
        .str.to_datetime()
    )
)

weather_df = (
    weather_df
    .with_columns(
        pl.col("datetime")
        .str.to_datetime()
    )
)

holidays_df = (
    holidays_df
    .with_columns(
        pl.col("date")
        .str.to_date()
    )
)

# Rental Dataset Standardization
The direct and registered rentals are standardized into a common schema before merging.

In [13]:
direct_rentals = (
    direct_rentals

    .with_columns([

        pl.col("datetime")
        .dt.truncate("1h")
        .alias("datetime_hour"),

        pl.lit(1)
        .alias("rental_type_direct")
    ])

    .select([
        "id",
        "datetime",
        "datetime_hour",
        "location_id",
        "user_id",
        "rental_type_direct"
    ])
)


registered_rentals = (
    registered_rentals

    .with_columns([

        pl.col("datetime")
        .dt.truncate("1h")
        .alias("datetime_hour"),

        pl.lit(0)
        .alias("rental_type_direct")
    ])

    .select([
        "id",
        "datetime",
        "datetime_hour",
        "location_id",
        "user_id",
        "rental_type_direct"
    ])
)

# Unified Rental Dataset

Both rental datasets are vertically concatenated into a single dataset.

In [15]:
unified_rentals = pl.concat([
    direct_rentals,
    registered_rentals
])

# Hourly Aggregation by Location

The unified rentals dataset is aggregated at:

* hourly granularity
* per location

This creates the core analytical dataset.


In [16]:
hourly_rentals_by_location = (

    unified_rentals

    .group_by([
        "datetime_hour",
        "location_id"
    ])

    .agg([

        pl.len()
        .alias("rental_count"),

        (
            pl.col("rental_type_direct") == 1
        )
        .sum()
        .alias("direct_rentals"),

        (
            pl.col("rental_type_direct") == 0
        )
        .sum()
        .alias("registered_rentals"),

        pl.col("user_id")
        .n_unique()
        .alias("unique_users")
    ])

    .sort([
        "datetime_hour",
        "location_id"
    ])
)

Weather Data Preparation
Weather data is standardized to hourly granularity to match the rental aggregation level.

In [19]:
weather_hourly = (
    weather_df
    .with_columns(
        pl.col("datetime")
        .dt.truncate("1h")
        .alias("datetime_hour")
    )
    .select([
        "datetime_hour",
        "conditions",
        "temperature_c",
        "perceived_temperature_c",
        "humidity",
        "windspeed_kmh"
    ])
)

# Join Weather Data

A left join is performed so all rental records are preserved even if weather data is missing.

In [20]:
rentals_with_weather = (

    hourly_rentals_by_location

    .join(
        weather_hourly,
        on="datetime_hour",
        how="left"
    )
)

# Holiday Data Preparation

Holiday data is prepared at daily granularity.

In [24]:
holidays_prepared = (

    holidays_df

    .select([
        "date",
        "holiday"
    ])
)

# Add Date Column

The hourly dataset requires a daily date column before joining holiday information.

In [21]:
rentals_with_weather = (

    rentals_with_weather

    .with_columns(

        pl.col("datetime_hour")
        .dt.date()
        .alias("date")
    )
)

# Join Holiday Data

A left join is performed using the date column.

In [25]:
final_df = (

    rentals_with_weather

    .join(
        holidays_prepared,
        on="date",
        how="left"
    )

    .with_columns(

        pl.col("holiday")
        .is_not_null()
        .cast(pl.Int8)
        .alias("is_holiday")
    )
)

# Final Dataset Validation

In [26]:
validate_dataframe(
    "Final Dataset",
    final_df
)


Final Dataset
--------------------------------------------------
Shape: (312386, 14)

Columns:
['datetime_hour', 'location_id', 'rental_count', 'direct_rentals', 'registered_rentals', 'unique_users', 'conditions', 'temperature_c', 'perceived_temperature_c', 'humidity', 'windspeed_kmh', 'date', 'holiday', 'is_holiday']

Null Counts:
shape: (1, 14)
┌────────────┬────────────┬────────────┬────────────┬───┬────────────┬──────┬─────────┬────────────┐
│ datetime_h ┆ location_i ┆ rental_cou ┆ direct_ren ┆ … ┆ windspeed_ ┆ date ┆ holiday ┆ is_holiday │
│ our        ┆ d          ┆ nt         ┆ tals       ┆   ┆ kmh        ┆ ---  ┆ ---     ┆ ---        │
│ ---        ┆ ---        ┆ ---        ┆ ---        ┆   ┆ ---        ┆ u32  ┆ u32     ┆ u32        │
│ u32        ┆ u32        ┆ u32        ┆ u32        ┆   ┆ u32        ┆      ┆         ┆            │
╞════════════╪════════════╪════════════╪════════════╪═══╪════════════╪══════╪═════════╪════════════╡
│ 0          ┆ 0          ┆ 0          ┆ 0  

# Merged Dataset

The final dataset contains:

* hourly rental demand
* direct vs registered rentals
* unique users
* weather conditions
* holiday indicators


In [27]:
final_df.schema

Schema([('datetime_hour', Datetime(time_unit='us', time_zone=None)),
        ('location_id', Int64),
        ('rental_count', UInt32),
        ('direct_rentals', UInt32),
        ('registered_rentals', UInt32),
        ('unique_users', UInt32),
        ('conditions', String),
        ('temperature_c', Float64),
        ('perceived_temperature_c', Float64),
        ('humidity', Float64),
        ('windspeed_kmh', Float64),
        ('date', Date),
        ('holiday', String),
        ('is_holiday', Int8)])

# Create Machine Learning Ready Dataset

The final analytical dataset is further enriched with time-based engineered features for machine learning workflows.

These features help machine learning models better understand cyclical temporal patterns such as:

* hours in a day
* weekdays in a week
* months in a year
* weekends
* seasonal behavior

Cyclical encoding is important because time values are naturally circular.

For example:

* hour `23` is close to hour `0`
* December is close to January

Using sine and cosine transformations preserves these cyclical relationships.

In [28]:
import math

# Generate Cyclical and Seasonal Features

The following transformations are applied:

* hour cyclical encoding
* weekday cyclical encoding
* month cyclical encoding
* weekend indicator
* seasonal categorization

The unused `date` column is removed at the end.

In [29]:
ml_ready_dataset = (

    final_df

    .with_columns([

        # ---------------------------------------------
        # HOUR CYCLICAL FEATURES
        # ---------------------------------------------

        (
                (
                        2 * math.pi *
                        pl.col("datetime_hour").dt.hour()
                ) / 24
        )
        .sin()
        .alias("hour_sin"),

        (
                (
                        2 * math.pi *
                        pl.col("datetime_hour").dt.hour()
                ) / 24
        )
        .cos()
        .alias("hour_cos"),

        # ---------------------------------------------
        # WEEKDAY CYCLICAL FEATURES
        # ---------------------------------------------

        (
                (
                        2 * math.pi *
                        pl.col("datetime_hour").dt.weekday()
                ) / 7
        )
        .sin()
        .alias("weekday_sin"),

        (
                (
                        2 * math.pi *
                        pl.col("datetime_hour").dt.weekday()
                ) / 7
        )
        .cos()
        .alias("weekday_cos"),

        # ---------------------------------------------
        # MONTH CYCLICAL FEATURES
        # ---------------------------------------------

        (
                (
                        2 * math.pi *
                        pl.col("datetime_hour").dt.month()
                ) / 12
        )
        .sin()
        .alias("month_sin"),

        (
                (
                        2 * math.pi *
                        pl.col("datetime_hour").dt.month()
                ) / 12
        )
        .cos()
        .alias("month_cos"),

        # ---------------------------------------------
        # WEEKEND FEATURE
        # ---------------------------------------------

        (
                pl.col("datetime_hour")
                .dt.weekday() >= 5
        )
        .cast(pl.Int8)
        .alias("is_weekend"),

        # ---------------------------------------------
        # SEASON FEATURE
        # ---------------------------------------------

        pl.when(
            pl.col("datetime_hour")
            .dt.month()
            .is_in([12, 1, 2])
        )
        .then(pl.lit("Winter"))

        .when(
            pl.col("datetime_hour")
            .dt.month()
            .is_in([3, 4, 5])
        )
        .then(pl.lit("Spring"))

        .when(
            pl.col("datetime_hour")
            .dt.month()
            .is_in([6, 7, 8])
        )
        .then(pl.lit("Summer"))

        .otherwise(pl.lit("Fall"))
        .alias("season")
    ])

    .drop([
        "date"
    ])
)


# Feature Engineering Summary

The engineered features provide additional temporal context for machine learning models.

## Hour Cyclical Features

* `hour_sin`
* `hour_cos`

Encodes the hour of day while preserving cyclical relationships.

---

## Weekday Cyclical Features

* `weekday_sin`
* `weekday_cos`

Encodes the day of week cyclically.

---

## Month Cyclical Features

* `month_sin`
* `month_cos`

Encodes yearly seasonal progression.

---

## Weekend Indicator

* `is_weekend`

Binary feature:

* `1` = weekend
* `0` = weekday

---

## Season Feature

* `season`

Categorical seasonal label:

* Winter
* Spring
* Summer
* Fall

---

# Markdown Cell

# Conclusion

The dataset is now fully prepared for machine learning workflows.

The final dataset contains:

* rental demand metrics
* weather information
* holiday indicators
* cyclical temporal features
* seasonal features

This dataset can now be used for:

* demand forecasting
* predictive modeling
* anomaly detection
* time series analysis
* feature experimentation

In [33]:
ml_ready_dataset.head()

datetime_hour,location_id,rental_count,direct_rentals,registered_rentals,unique_users,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh,holiday,is_holiday,hour_sin,hour_cos,weekday_sin,weekday_cos,month_sin,month_cos,is_weekend,season
datetime[μs],i64,u32,u32,u32,u32,str,f64,f64,f64,f64,str,i8,f64,f64,f64,f64,f64,f64,i8,str
2011-01-01 00:00:00,2,1,1,0,1,"""clear""",3.3,3.0,81.0,0.0,null,0,0.0,1.0,-0.781831,0.62349,0.5,0.866025,1,"""Winter"""
2011-01-01 00:00:00,3,1,0,1,1,"""clear""",3.3,3.0,81.0,0.0,null,0,0.0,1.0,-0.781831,0.62349,0.5,0.866025,1,"""Winter"""
2011-01-01 00:00:00,4,2,0,2,2,"""clear""",3.3,3.0,81.0,0.0,null,0,0.0,1.0,-0.781831,0.62349,0.5,0.866025,1,"""Winter"""
2011-01-01 00:00:00,5,1,1,0,1,"""clear""",3.3,3.0,81.0,0.0,null,0,0.0,1.0,-0.781831,0.62349,0.5,0.866025,1,"""Winter"""
2011-01-01 00:00:00,9,1,0,1,1,"""clear""",3.3,3.0,81.0,0.0,null,0,0.0,1.0,-0.781831,0.62349,0.5,0.866025,1,"""Winter"""


In [36]:
print(ml_ready_dataset.shape)

(312386, 21)


In [37]:
ml_ready_dataset.schema

Schema([('datetime_hour', Datetime(time_unit='us', time_zone=None)),
        ('location_id', Int64),
        ('rental_count', UInt32),
        ('direct_rentals', UInt32),
        ('registered_rentals', UInt32),
        ('unique_users', UInt32),
        ('conditions', String),
        ('temperature_c', Float64),
        ('perceived_temperature_c', Float64),
        ('humidity', Float64),
        ('windspeed_kmh', Float64),
        ('holiday', String),
        ('is_holiday', Int8),
        ('hour_sin', Float64),
        ('hour_cos', Float64),
        ('weekday_sin', Float64),
        ('weekday_cos', Float64),
        ('month_sin', Float64),
        ('month_cos', Float64),
        ('is_weekend', Int8),
        ('season', String)])